# Google Play Rating-Derived Weak Sentiment Baseline v0

This notebook creates a lightweight downstream sentiment baseline from the validated **Feature Engineering v0** review table.

The scope follows the current project requirements:

- derive weak sentiment labels from source ratings:
  - 1–2 stars = `negative`
  - 3 stars = `neutral`
  - 4–5 stars = `positive`
- show class distribution
- compare several feature patterns across the three groups
- create a clean modeling-ready dataset
- test one simple and interpretable baseline model
- exclude rating and rating-derived fields from model inputs
- document that the target is a weak label rather than a manually verified sentiment label
- keep the work exploratory rather than production-oriented

The notebook does **not** build a complex NLP model or claim that the generated labels are ground-truth sentiment.

## 1. Imports and configuration

The baseline uses:

- pandas and NumPy for data preparation
- scikit-learn for a group-aware train/test split, TF-IDF features, and one linear classifier
- matplotlib for simple exploratory figures
- the Python standard library for hashing, metadata, and output validation

The random seed and all main modeling choices are fixed for reproducibility.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
LABEL_ORDER = ["negative", "neutral", "positive"]
LABEL_MAP = {
    1: "negative",
    2: "negative",
    3: "neutral",
    4: "positive",
    5: "positive",
}

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORT_DIR = REPO_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = OUTPUT_DIR / "review_features_v0.csv"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_obj:
        while True:
            chunk = file_obj.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

print("Repository root:", REPO_ROOT)
print("Input path:", INPUT_PATH)
print("Random state:", RANDOM_STATE)

Repository root: /mnt/data/google_play_weak_sentiment_baseline_v0_complete_github_package
Input path: /mnt/data/google_play_weak_sentiment_baseline_v0_complete_github_package/outputs/review_features_v0.csv
Random state: 42


## 2. Load and validate the Feature Engineering v0 input

This downstream notebook reads `outputs/review_features_v0.csv`, which was generated from the validated cleaned-review database.

Before any label or model work, the notebook checks:

- file existence and SHA-256
- expected row count
- required source and feature columns
- unique review keys
- valid 1–5 rating values
- complete app coverage
- absence of duplicate persisted review identities

In [2]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        "review_features_v0.csv was not found. Run Feature Engineering v0 first "
        "or place its output in the repository outputs/ directory."
    )

required_columns = [
    "review_key",
    "source",
    "app_id",
    "app_name",
    "review_id",
    "content_cleaned",
    "score",
    "rating_group",
    "low_rating_flag",
    "review_char_count",
    "review_word_count",
    "alphanumeric_char_count",
    "short_review_flag",
    "low_signal_flag",
    "duplicate_identity_flag",
    "same_app_text_frequency",
    "repeated_text_flag",
    "language_group",
    "non_english_flag",
    "has_developer_reply",
    "review_created_at",
    "review_date",
    "fetched_at",
    "collection_date",
    "review_age_hours_at_collection",
    "app_version",
    "run_id",
    "issue_crash_bug_flag",
    "issue_performance_loading_flag",
    "issue_login_account_flag",
    "issue_payment_billing_flag",
    "issue_ads_flag",
    "issue_update_version_flag",
    "issue_support_service_flag",
    "issue_indicator_count",
    "any_issue_indicator_flag",
]

reviews = pd.read_csv(INPUT_PATH)
missing_columns = sorted(set(required_columns) - set(reviews.columns))

source_validation = pd.DataFrame(
    [
        {"check": "input_file_exists", "result": INPUT_PATH.exists(), "value": str(INPUT_PATH)},
        {"check": "input_file_size_bytes", "result": INPUT_PATH.stat().st_size > 0, "value": INPUT_PATH.stat().st_size},
        {"check": "input_sha256", "result": True, "value": sha256_file(INPUT_PATH)},
        {"check": "expected_row_count_34601", "result": len(reviews) == 34601, "value": len(reviews)},
        {"check": "all_required_columns_present", "result": len(missing_columns) == 0, "value": ", ".join(missing_columns) or "none"},
        {"check": "review_key_complete", "result": reviews["review_key"].notna().all(), "value": int(reviews["review_key"].isna().sum())},
        {"check": "review_key_unique", "result": reviews["review_key"].is_unique, "value": int(reviews["review_key"].duplicated().sum())},
        {"check": "score_complete", "result": reviews["score"].notna().all(), "value": int(reviews["score"].isna().sum())},
        {"check": "score_domain_1_to_5", "result": reviews["score"].isin([1, 2, 3, 4, 5]).all(), "value": sorted(reviews["score"].dropna().unique().tolist())},
        {"check": "ten_apps_present", "result": reviews["app_name"].nunique() == 10, "value": reviews["app_name"].nunique()},
        {"check": "no_duplicate_identity_flags", "result": int(reviews["duplicate_identity_flag"].sum()) == 0, "value": int(reviews["duplicate_identity_flag"].sum())},
    ]
)

source_validation.to_csv(OUTPUT_DIR / "weak_sentiment_source_validation_v0.csv", index=False)

if not source_validation["result"].all():
    failed = source_validation.loc[~source_validation["result"]]
    raise ValueError("Source validation failed:\n" + failed.to_string(index=False))

print(f"Loaded {len(reviews):,} review rows across {reviews['app_name'].nunique()} apps.")
print("Input SHA-256:", sha256_file(INPUT_PATH))
display(source_validation)

Loaded 34,601 review rows across 10 apps.
Input SHA-256: 7e65a3d282b7121012a940c931bed0a10f32ebf83d8744d1021545dbb87fe2fc


,check,result,value
0,input_file_exists,True,/mnt/data/google_play_weak_sentiment_baseline_...
1,input_file_size_bytes,True,15447118
2,input_sha256,True,7e65a3d282b7121012a940c931bed0a10f32ebf83d8744...
3,expected_row_count_34601,True,34601
4,all_required_columns_present,True,none
5,review_key_complete,True,0
6,review_key_unique,True,0
7,score_complete,True,0
8,score_domain_1_to_5,True,"[1, 2, 3, 4, 5]"
9,ten_apps_present,True,10


## 3. Create rating-derived weak sentiment labels

The target is generated deterministically:

| Source rating | Weak sentiment label |
|---|---|
| 1–2 | `negative` |
| 3 | `neutral` |
| 4–5 | `positive` |

These labels are **weak labels**. They are not manually reviewed sentiment annotations. A numeric rating may not fully match the tone or meaning of the written review.

In [3]:
reviews["weak_sentiment_label"] = reviews["score"].map(LABEL_MAP)

if reviews["weak_sentiment_label"].isna().any():
    raise ValueError("At least one score could not be mapped to a weak sentiment label.")

class_distribution = (
    reviews["weak_sentiment_label"]
    .value_counts()
    .reindex(LABEL_ORDER)
    .rename_axis("weak_sentiment_label")
    .reset_index(name="review_count")
)
class_distribution["review_share"] = class_distribution["review_count"] / len(reviews)
class_distribution["review_share_pct"] = class_distribution["review_share"] * 100

class_distribution.to_csv(
    OUTPUT_DIR / "weak_sentiment_class_distribution_v0.csv",
    index=False,
)

display(class_distribution)

,weak_sentiment_label,review_count,review_share,review_share_pct
0,negative,9948,0.287506,28.750614
1,neutral,1629,0.047080,4.707956
2,positive,23024,0.665414,66.541429


## 4. Show class distribution

The dataset is imbalanced. Positive reviews are the majority, while three-star neutral reviews are a small class.

Because of this imbalance:

- raw accuracy alone is not sufficient
- macro F1 and balanced accuracy are reported
- the model uses class balancing
- neutral-class results require particular caution

In [4]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    class_distribution["weak_sentiment_label"],
    class_distribution["review_count"],
)
ax.set_title("Rating-Derived Weak Sentiment Class Distribution")
ax.set_xlabel("Weak sentiment label")
ax.set_ylabel("Review count")

for bar, count, share in zip(
    bars,
    class_distribution["review_count"],
    class_distribution["review_share_pct"],
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}\n({share:.1f}%)",
        ha="center",
        va="bottom",
    )

fig.tight_layout()
class_figure_path = FIGURE_DIR / "weak_sentiment_class_distribution_v0.png"
fig.savefig(class_figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", class_figure_path)

Saved: /mnt/data/google_play_weak_sentiment_baseline_v0_complete_github_package/reports/figures/weak_sentiment_class_distribution_v0.png


/tmp/ipykernel_1081/4023127541.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Compare feature patterns across the three weak-label groups

The comparison uses existing Feature Engineering v0 fields. It is descriptive rather than causal.

The main comparisons include:

- review length
- short-review and low-signal shares
- repeated-text share
- developer-reply availability
- keyword issue-indicator share
- language heuristic distribution

Developer-reply availability is included in EDA, but it is not used in the baseline model because replies may be added after the review and may reflect app-specific response practices.

In [5]:
feature_patterns = (
    reviews.groupby("weak_sentiment_label")
    .agg(
        review_count=("review_key", "size"),
        average_review_char_count=("review_char_count", "mean"),
        median_review_char_count=("review_char_count", "median"),
        average_review_word_count=("review_word_count", "mean"),
        median_review_word_count=("review_word_count", "median"),
        short_review_share=("short_review_flag", "mean"),
        low_signal_share=("low_signal_flag", "mean"),
        repeated_text_share=("repeated_text_flag", "mean"),
        developer_reply_share=("has_developer_reply", "mean"),
        any_issue_indicator_share=("any_issue_indicator_flag", "mean"),
        average_issue_indicator_count=("issue_indicator_count", "mean"),
        average_review_age_hours=("review_age_hours_at_collection", "mean"),
    )
    .reindex(LABEL_ORDER)
    .reset_index()
)

language_patterns = (
    reviews.groupby("weak_sentiment_label")["language_group"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(LABEL_ORDER)
    .reset_index()
)

for column in ["english_likely", "non_english_likely", "undetermined"]:
    if column not in language_patterns.columns:
        language_patterns[column] = 0.0

language_patterns = language_patterns.rename(
    columns={
        "english_likely": "english_likely_share",
        "non_english_likely": "non_english_likely_share",
        "undetermined": "language_undetermined_share",
    }
)

feature_patterns = feature_patterns.merge(
    language_patterns[
        [
            "weak_sentiment_label",
            "english_likely_share",
            "non_english_likely_share",
            "language_undetermined_share",
        ]
    ],
    on="weak_sentiment_label",
    how="left",
)

feature_patterns.to_csv(
    OUTPUT_DIR / "weak_sentiment_feature_patterns_v0.csv",
    index=False,
)

display(feature_patterns.round(4))

,weak_sentiment_label,review_count,average_review_char_count,median_review_char_count,average_review_word_count,median_review_word_count,short_review_share,low_signal_share,repeated_text_share,developer_reply_share,any_issue_indicator_share,average_issue_indicator_count,average_review_age_hours,english_likely_share,non_english_likely_share,language_undetermined_share
0,negative,9948,137.9657,84.0,25.5575,16.0,0.1624,0.1207,0.0461,0.1560,0.4273,0.5404,59.8862,0.8145,0.0299,0.1556
1,neutral,1629,105.6667,57.0,19.9982,11.0,0.2664,0.2142,0.1240,0.2106,0.3026,0.3671,53.0146,0.7287,0.0301,0.2413
2,positive,23024,38.4235,15.0,7.2259,3.0,0.5873,0.4926,0.3170,0.1103,0.0630,0.0713,50.2945,0.4621,0.0157,0.5222


In [6]:
issue_columns = [
    "issue_crash_bug_flag",
    "issue_performance_loading_flag",
    "issue_login_account_flag",
    "issue_payment_billing_flag",
    "issue_ads_flag",
    "issue_update_version_flag",
    "issue_support_service_flag",
]

issue_patterns = (
    reviews.groupby("weak_sentiment_label")[issue_columns]
    .mean()
    .reindex(LABEL_ORDER)
    .T
    .reset_index()
    .rename(columns={"index": "issue_indicator"})
)

issue_patterns.to_csv(
    OUTPUT_DIR / "weak_sentiment_issue_patterns_v0.csv",
    index=False,
)

app_counts = pd.crosstab(
    reviews["app_name"],
    reviews["weak_sentiment_label"],
).reindex(columns=LABEL_ORDER, fill_value=0)

app_shares = app_counts.div(app_counts.sum(axis=1), axis=0)
app_distribution = app_counts.add_suffix("_count").join(
    app_shares.add_suffix("_share")
)
app_distribution.insert(0, "review_count", app_counts.sum(axis=1))
app_distribution = app_distribution.reset_index()

app_distribution.to_csv(
    OUTPUT_DIR / "weak_sentiment_app_distribution_v0.csv",
    index=False,
)

display(issue_patterns.round(4))
display(app_distribution.round(4))

weak_sentiment_label,issue_indicator,negative,neutral,positive
0,issue_crash_bug_flag,0.0787,0.0485,0.0069
1,issue_performance_loading_flag,0.0398,0.0473,0.0050
2,issue_login_account_flag,0.1244,0.0460,0.0136
3,issue_payment_billing_flag,0.0999,0.0890,0.0162
4,issue_ads_flag,0.0811,0.0632,0.0119
5,issue_update_version_flag,0.0701,0.0608,0.0132
6,issue_support_service_flag,0.0463,0.0123,0.0044


weak_sentiment_label,app_name,review_count,negative_count,neutral_count,positive_count,negative_share,neutral_share,positive_share
0,DoorDash,1975,873,125,977,0.4420,0.0633,0.4947
1,Duolingo,2098,160,95,1843,0.0763,0.0453,0.8785
2,Google Maps,2635,833,150,1652,0.3161,0.0569,0.6269
3,Instagram,6045,1585,212,4248,0.2622,0.0351,0.7027
4,Netflix,2365,796,120,1449,0.3366,0.0507,0.6127
5,Reddit,1991,963,57,971,0.4837,0.0286,0.4877
6,Spotify,4174,966,272,2936,0.2314,0.0652,0.7034
7,TikTok,4059,1046,228,2785,0.2577,0.0562,0.6861
8,Uber,3440,929,97,2414,0.2701,0.0282,0.7017
9,YouTube,5819,1797,273,3749,0.3088,0.0469,0.6443


In [7]:
rate_columns = [
    "short_review_share",
    "low_signal_share",
    "repeated_text_share",
    "developer_reply_share",
    "any_issue_indicator_share",
]

plot_data = feature_patterns.set_index("weak_sentiment_label")[rate_columns].T

fig, ax = plt.subplots(figsize=(10, 6))
plot_data.plot(kind="bar", ax=ax)
ax.set_title("Selected Feature Rates by Weak Sentiment Group")
ax.set_xlabel("Feature")
ax.set_ylabel("Share of reviews")
ax.set_ylim(0, 0.70)
ax.tick_params(axis="x", rotation=30)
ax.legend(title="Weak label")
fig.tight_layout()

pattern_figure_path = FIGURE_DIR / "weak_sentiment_feature_patterns_v0.png"
fig.savefig(pattern_figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", pattern_figure_path)

Saved: /mnt/data/google_play_weak_sentiment_baseline_v0_complete_github_package/reports/figures/weak_sentiment_feature_patterns_v0.png


/tmp/ipykernel_1081/1669560909.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Leakage controls and modeling scope

The weak target is created from `score`, so the following fields are explicitly excluded from model inputs:

- `score`
- `rating_group`
- `low_rating_flag`

Additional controls:

- stable IDs are retained only for traceability
- app identity is retained in the modeling-ready output but excluded from the model, so the classifier does not simply learn app-specific rating composition
- developer-reply availability is excluded from the model because it may be a post-review operational signal
- collection dates, run IDs, app versions, and source-lag fields are excluded to avoid collection-batch shortcuts
- identical normalized review text is kept within only one train/test partition
- explicit written star-rating expressions are replaced with a generic `ratingmention` token before modeling, so text such as “1 star” does not directly expose the derived class value

The text redaction is narrow and transparent. It does not attempt to rewrite the review or remove ordinary sentiment language.

In [8]:
reviews["content_cleaned"] = reviews["content_cleaned"].fillna("").astype(str)

rating_mention_pattern = re.compile(
    r"(?:"
    r"\b[1-5]\s*[- ]?\s*stars?\b"
    r"|\b(?:one|two|three|four|five)\s*[- ]?\s*stars?\b"
    r"|[⭐★]{1,5}"
    r")",
    flags=re.IGNORECASE,
)

reviews["rating_mention_redacted_flag"] = (
    reviews["content_cleaned"].str.contains(rating_mention_pattern, regex=True).astype(int)
)

reviews["model_text"] = (
    reviews["content_cleaned"]
    .str.replace(rating_mention_pattern, " ratingmention ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

normalized_text = (
    reviews["content_cleaned"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

empty_text_mask = normalized_text.eq("")
normalized_text.loc[empty_text_mask] = (
    "__empty__" + reviews.loc[empty_text_mask, "review_key"].astype(str)
)

reviews["text_group_id"] = normalized_text.map(
    lambda value: hashlib.sha256(value.encode("utf-8")).hexdigest()
)

leakage_excluded_fields = [
    "score",
    "rating_group",
    "low_rating_flag",
]

print(
    "Reviews with explicit star-rating expressions redacted:",
    f"{reviews['rating_mention_redacted_flag'].sum():,}",
)
print("Direct rating-derived fields excluded:", leakage_excluded_fields)

Reviews with explicit star-rating expressions redacted: 415
Direct rating-derived fields excluded: ['score', 'rating_group', 'low_rating_flag']


## 7. Create a group-aware train/test split

A five-fold `StratifiedGroupKFold` object is used to create one deterministic 80/20 holdout split.

Grouping is based on normalized review text, so an identical text string cannot appear in both training and test data. This is stricter than splitting only by review ID and helps reduce inflated performance from repeated generic reviews.

In [9]:
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

train_indices, test_indices = next(
    splitter.split(
        reviews,
        y=reviews["weak_sentiment_label"],
        groups=reviews["text_group_id"],
    )
)

reviews["dataset_split"] = "train"
reviews.loc[test_indices, "dataset_split"] = "test"

train_df = reviews.loc[train_indices].copy()
test_df = reviews.loc[test_indices].copy()

text_group_overlap = len(
    set(train_df["text_group_id"]).intersection(set(test_df["text_group_id"]))
)
review_key_overlap = len(
    set(train_df["review_key"]).intersection(set(test_df["review_key"]))
)

split_summary = (
    reviews.groupby(["dataset_split", "weak_sentiment_label"])
    .size()
    .rename("review_count")
    .reset_index()
)
split_summary["split_share"] = split_summary.groupby("dataset_split")[
    "review_count"
].transform(lambda values: values / values.sum())

print(f"Training rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print("Text-group overlap:", text_group_overlap)
print("Review-key overlap:", review_key_overlap)
display(split_summary)

Training rows: 27,681
Test rows: 6,920
Text-group overlap: 0
Review-key overlap: 0


,dataset_split,weak_sentiment_label,review_count,split_share
0,test,negative,1990,0.287572
1,test,neutral,325,0.046965
2,test,positive,4605,0.665462
3,train,negative,7958,0.287490
4,train,neutral,1304,0.047108
5,train,positive,18419,0.665402


## 8. Export a clean modeling-ready dataset

The modeling-ready file contains:

- traceability fields
- the rating-derived weak target
- deterministic train/test assignment
- redacted model text
- selected review-level features

It does **not** contain `score`, `rating_group`, or `low_rating_flag`.

`app_id` and `app_name` remain for auditing and group summaries, but they are not passed to the model.

In [10]:
trace_columns = [
    "review_key",
    "app_id",
    "app_name",
    "weak_sentiment_label",
    "dataset_split",
    "text_group_id",
]

model_feature_columns = [
    "model_text",
    "rating_mention_redacted_flag",
    "review_char_count",
    "review_word_count",
    "alphanumeric_char_count",
    "same_app_text_frequency",
    "issue_indicator_count",
    "short_review_flag",
    "low_signal_flag",
    "repeated_text_flag",
    "language_group",
    "issue_crash_bug_flag",
    "issue_performance_loading_flag",
    "issue_login_account_flag",
    "issue_payment_billing_flag",
    "issue_ads_flag",
    "issue_update_version_flag",
    "issue_support_service_flag",
    "any_issue_indicator_flag",
]

modeling_ready = reviews[trace_columns + model_feature_columns].copy()

for forbidden_column in leakage_excluded_fields:
    if forbidden_column in modeling_ready.columns:
        raise ValueError(f"Leakage field found in modeling-ready data: {forbidden_column}")

modeling_ready_path = OUTPUT_DIR / "modeling_ready_weak_sentiment_v0.csv"
modeling_ready.to_csv(modeling_ready_path, index=False)

print("Modeling-ready shape:", modeling_ready.shape)
print("Saved:", modeling_ready_path)
display(modeling_ready.head(5))

Modeling-ready shape: (34601, 25)
Saved: /mnt/data/google_play_weak_sentiment_baseline_v0_complete_github_package/outputs/modeling_ready_weak_sentiment_v0.csv


,review_key,app_id,app_name,weak_sentiment_label,dataset_split,text_group_id,model_text,rating_mention_redacted_flag,review_char_count,review_word_count,alphanumeric_char_count,same_app_text_frequency,issue_indicator_count,short_review_flag,low_signal_flag,repeated_text_flag,language_group,issue_crash_bug_flag,issue_performance_loading_flag,issue_login_account_flag,issue_payment_billing_flag,issue_ads_flag,issue_update_version_flag,issue_support_service_flag,any_issue_indicator_flag
0,8700e0bccfc44936acc518dab7ffa6ce9aae9dfe5cd14b...,com.dd.doordash,DoorDash,negative,train,c26c30488ca0d2faefd5dfda293a497d3b6eb635592857...,They now charge for a Regular delivery 2.99 or...,0,125,25,99,1,1,0,0,0,english_likely,0,0,0,1,0,0,0,1
1,f40e0f6bd214601508a541dfbf7e6039d7857433844f83...,com.dd.doordash,DoorDash,negative,test,89ac8e68595ca75615012110e5364255f23d0b39ea5044...,Update: This app STILL deserves ratingmention ...,1,415,72,324,1,3,0,0,0,english_likely,0,0,1,0,0,1,1,1
2,0774835d100ffc9bfa3b14e1f6195708721493fcb3010c...,com.dd.doordash,DoorDash,negative,train,5ba26b52ccc806576fbc89a1475c909bdea8700abe2356...,Over priced app that always crashes. Use any o...,0,81,15,64,1,1,0,0,0,english_likely,1,0,0,0,0,0,0,1
3,02e837ed23ecc2c4cb4eb0e862fb856e7743df6bcdca5b...,com.dd.doordash,DoorDash,negative,train,74b12371fa4b35c14f44b40ff4ce719944ea1b6d7e8f58...,I keep getting an error message and I getting ...,0,56,11,46,1,1,0,0,0,english_likely,1,0,0,0,0,0,0,1
4,660e98ecca88ba5086075447c3122900c6c22b3eb57d6b...,com.dd.doordash,DoorDash,negative,train,e06da30434b904a063a13413df8e2f14d71645168515d4...,trash,0,5,1,5,2,0,1,1,1,undetermined,0,0,0,0,0,0,0,0


## 9. Define one transparent baseline model

The baseline is a **class-weighted linear support vector classifier** (`LinearSVC`).

Why this model:

- it is linear and interpretable through feature coefficients
- it works well with sparse TF-IDF text
- it does not require a complex neural or transformer architecture
- class weighting helps reduce the effect of the small neutral class

Model inputs:

- TF-IDF word unigrams and bigrams from `model_text`
- deterministic text-length fields
- short, low-signal, repeated-text, and issue indicator fields
- language heuristic group

The model uses fixed default regularization (`C=1.0`) and no hyperparameter search. A majority-class dummy classifier is included only as a reference point.

In [11]:
text_column = "model_text"

numeric_columns = [
    "review_char_count",
    "review_word_count",
    "alphanumeric_char_count",
    "same_app_text_frequency",
    "issue_indicator_count",
]

binary_columns = [
    "rating_mention_redacted_flag",
    "short_review_flag",
    "low_signal_flag",
    "repeated_text_flag",
    "issue_crash_bug_flag",
    "issue_performance_loading_flag",
    "issue_login_account_flag",
    "issue_payment_billing_flag",
    "issue_ads_flag",
    "issue_update_version_flag",
    "issue_support_service_flag",
    "any_issue_indicator_flag",
]

categorical_columns = ["language_group"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                ngram_range=(1, 2),
                min_df=5,
                max_df=0.98,
                max_features=6000,
                sublinear_tf=True,
            ),
            text_column,
        ),
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_columns,
        ),
        (
            "binary",
            SimpleImputer(strategy="most_frequent"),
            binary_columns,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

baseline_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            LinearSVC(
                C=1.0,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

baseline_model.fit(
    train_df,
    train_df["weak_sentiment_label"],
)

test_predictions = baseline_model.predict(test_df)

dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(
    np.zeros((len(train_df), 1)),
    train_df["weak_sentiment_label"],
)
dummy_predictions = dummy_model.predict(np.zeros((len(test_df), 1)))

print("Baseline model fitted.")
print("Transformed feature count:", len(
    baseline_model.named_steps["preprocess"].get_feature_names_out()
))

Baseline model fitted.
Transformed feature count: 6020


## 10. Evaluate the baseline

The evaluation reports:

- accuracy
- balanced accuracy
- macro F1
- weighted F1
- class-specific precision, recall, and F1
- confusion matrix

Macro F1 and balanced accuracy receive emphasis because the neutral class represents less than 5% of the data.

In [12]:
def metric_row(model_name, y_true, y_pred):
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "test_rows": len(y_true),
    }

model_metrics = pd.DataFrame(
    [
        metric_row(
            "majority_class_reference",
            test_df["weak_sentiment_label"],
            dummy_predictions,
        ),
        metric_row(
            "class_weighted_linear_svc",
            test_df["weak_sentiment_label"],
            test_predictions,
        ),
    ]
)

model_metrics.to_csv(
    OUTPUT_DIR / "weak_sentiment_model_metrics_v0.csv",
    index=False,
)

report_dict = classification_report(
    test_df["weak_sentiment_label"],
    test_predictions,
    labels=LABEL_ORDER,
    output_dict=True,
    zero_division=0,
)

classification_rows = []
for label in LABEL_ORDER:
    classification_rows.append(
        {
            "class": label,
            "precision": report_dict[label]["precision"],
            "recall": report_dict[label]["recall"],
            "f1_score": report_dict[label]["f1-score"],
            "support": int(report_dict[label]["support"]),
        }
    )

classification_rows.extend(
    [
        {
            "class": "macro_average",
            "precision": report_dict["macro avg"]["precision"],
            "recall": report_dict["macro avg"]["recall"],
            "f1_score": report_dict["macro avg"]["f1-score"],
            "support": int(report_dict["macro avg"]["support"]),
        },
        {
            "class": "weighted_average",
            "precision": report_dict["weighted avg"]["precision"],
            "recall": report_dict["weighted avg"]["recall"],
            "f1_score": report_dict["weighted avg"]["f1-score"],
            "support": int(report_dict["weighted avg"]["support"]),
        },
    ]
)

classification_report_df = pd.DataFrame(classification_rows)
classification_report_df.to_csv(
    OUTPUT_DIR / "weak_sentiment_classification_report_v0.csv",
    index=False,
)

confusion = confusion_matrix(
    test_df["weak_sentiment_label"],
    test_predictions,
    labels=LABEL_ORDER,
)
confusion_df = pd.DataFrame(
    confusion,
    index=[f"actual_{label}" for label in LABEL_ORDER],
    columns=[f"predicted_{label}" for label in LABEL_ORDER],
)
confusion_df.to_csv(
    OUTPUT_DIR / "weak_sentiment_confusion_matrix_v0.csv"
)

display(model_metrics.round(4))
display(classification_report_df.round(4))
display(confusion_df)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,test_rows
0,majority_class_reference,0.6655,0.3333,0.2664,0.5318,6920
1,class_weighted_linear_svc,0.8208,0.5867,0.5899,0.8196,6920


,class,precision,recall,f1_score,support
0,negative,0.7796,0.7412,0.7599,1990
1,neutral,0.1153,0.1138,0.1146,325
2,positive,0.8855,0.9051,0.8952,4605
3,macro_average,0.5935,0.5867,0.5899,6920
4,weighted_average,0.8189,0.8208,0.8196,6920


,predicted_negative,predicted_neutral,predicted_positive
actual_negative,1475,144,371
actual_neutral,120,37,168
actual_positive,297,140,4168


In [13]:
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(confusion)
ax.set_title("Weak Sentiment Baseline Confusion Matrix")
ax.set_xlabel("Predicted weak label")
ax.set_ylabel("Actual weak label")
ax.set_xticks(range(len(LABEL_ORDER)), LABEL_ORDER)
ax.set_yticks(range(len(LABEL_ORDER)), LABEL_ORDER)

for row_index in range(confusion.shape[0]):
    for column_index in range(confusion.shape[1]):
        ax.text(
            column_index,
            row_index,
            f"{confusion[row_index, column_index]:,}",
            ha="center",
            va="center",
        )

fig.colorbar(image, ax=ax)
fig.tight_layout()

confusion_figure_path = FIGURE_DIR / "weak_sentiment_confusion_matrix_v0.png"
fig.savefig(confusion_figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", confusion_figure_path)

Saved: /mnt/data/google_play_weak_sentiment_baseline_v0_complete_github_package/reports/figures/weak_sentiment_confusion_matrix_v0.png


/tmp/ipykernel_1081/3530349652.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Inspect linear model coefficients

For each class, the notebook exports the strongest positive and negative coefficient associations.

These coefficients help explain what the linear classifier used, but they are not causal findings. They can reflect:

- common sentiment wording
- app-review vocabulary
- weak-label noise
- class imbalance
- correlations created by the sampled apps and collection window

Neutral-class coefficients are expected to be less stable because the neutral class is both small and conceptually ambiguous.

In [14]:
feature_names = baseline_model.named_steps["preprocess"].get_feature_names_out()
model_step = baseline_model.named_steps["model"]

coefficient_rows = []
top_n = 20

for class_index, class_label in enumerate(model_step.classes_):
    class_coefficients = model_step.coef_[class_index]

    positive_indices = np.argsort(class_coefficients)[-top_n:][::-1]
    negative_indices = np.argsort(class_coefficients)[:top_n]

    for rank, feature_index in enumerate(positive_indices, start=1):
        coefficient_rows.append(
            {
                "class": class_label,
                "direction": "supports_class",
                "rank": rank,
                "feature": feature_names[feature_index],
                "coefficient": class_coefficients[feature_index],
            }
        )

    for rank, feature_index in enumerate(negative_indices, start=1):
        coefficient_rows.append(
            {
                "class": class_label,
                "direction": "opposes_class",
                "rank": rank,
                "feature": feature_names[feature_index],
                "coefficient": class_coefficients[feature_index],
            }
        )

top_coefficients = pd.DataFrame(coefficient_rows)
top_coefficients.to_csv(
    OUTPUT_DIR / "weak_sentiment_top_coefficients_v0.csv",
    index=False,
)

display(
    top_coefficients[
        top_coefficients["direction"].eq("supports_class")
    ].groupby("class").head(10)
)

,class,direction,rank,feature,coefficient
0,negative,supports_class,1,text__horrible,2.970245
1,negative,supports_class,2,text__worst,2.541554
2,negative,supports_class,3,text__garbage,2.518463
3,negative,supports_class,4,text__ruined,2.370571
4,negative,supports_class,5,text__trash,2.162767
5,negative,supports_class,6,text__stupid,2.130774
6,negative,supports_class,7,text__suddenly,2.108412
7,negative,supports_class,8,text__forcing,2.084942
8,negative,supports_class,9,text__greedy,2.050569
9,negative,supports_class,10,text__disgusting,2.046628


## 12. Save a small prediction-review sample

The sample prioritizes misclassified test reviews and then adds correctly classified reviews. It is intended for manual error review, not for estimating performance.

In [15]:
prediction_review = test_df[
    [
        "review_key",
        "app_name",
        "model_text",
        "weak_sentiment_label",
        "rating_mention_redacted_flag",
        "review_word_count",
        "low_signal_flag",
        "language_group",
        "any_issue_indicator_flag",
    ]
].copy()

prediction_review["predicted_weak_sentiment_label"] = test_predictions
prediction_review["correct_prediction"] = (
    prediction_review["weak_sentiment_label"]
    == prediction_review["predicted_weak_sentiment_label"]
).astype(int)

error_sample = (
    prediction_review[prediction_review["correct_prediction"].eq(0)]
    .sort_values(["weak_sentiment_label", "review_key"])
    .groupby("weak_sentiment_label", group_keys=False)
    .head(15)
)

correct_sample = (
    prediction_review[prediction_review["correct_prediction"].eq(1)]
    .sort_values(["weak_sentiment_label", "review_key"])
    .groupby("weak_sentiment_label", group_keys=False)
    .head(5)
)

prediction_sample = (
    pd.concat([error_sample, correct_sample], ignore_index=True)
    .sort_values(
        ["weak_sentiment_label", "correct_prediction", "review_key"],
        ascending=[True, True, True],
    )
    .reset_index(drop=True)
)

prediction_sample.to_csv(
    OUTPUT_DIR / "weak_sentiment_test_predictions_sample_v0.csv",
    index=False,
)

print("Prediction review sample rows:", len(prediction_sample))
display(prediction_sample.head(12))

Prediction review sample rows: 60


,review_key,app_name,model_text,weak_sentiment_label,rating_mention_redacted_flag,review_word_count,low_signal_flag,language_group,any_issue_indicator_flag,predicted_weak_sentiment_label,correct_prediction
0,0048b2bf40b00ffbd3d7932731a6f0b40b4e5c88fa631e...,YouTube,can you add some update like TikTok? Like stat...,negative,0,12,0,english_likely,1,neutral,0
1,0135c06c42840a6052a4716768425f033f1975683dc4cf...,Spotify,shuffle is actually the worst thing ever. I've...,negative,0,78,0,english_likely,0,positive,0
2,020fd0e25189a26e71e1f39703f7dc50e639042f46a46d...,Spotify,a music app where you can't even PICK the musi...,negative,0,12,0,english_likely,0,positive,0
3,021ea30896afe1c55c11f8b9a3a41b2cc7ec93c33b7d32...,Google Maps,barish ke time pani kaha bhara hua hai wo bhi ...,negative,0,12,0,undetermined,0,positive,0
4,032aaac495cb0cdeb92f9d3425bd6b3001173ade8e4cb4...,YouTube,so many ads. 2 ads per minute is the new stand...,negative,0,13,0,english_likely,1,neutral,0
5,0377cd91f0130966bc0046a01a8f64a8be89a16b151f84...,Instagram,lags sometimes,negative,0,2,1,undetermined,1,neutral,0
6,04d1dbf42a90bb4eed4d22daaed2421792d8a4f9397e2e...,YouTube,Good continue YouTube,negative,0,3,0,english_likely,0,positive,0
7,057885aaa69b97f8b3a3c21964fd140b83db8e62aa9415...,Instagram,a lot of disabled,negative,0,4,0,english_likely,0,positive,0
8,05b10387ad46f86790e6aec764bcc1e2f056aaa16a5a89...,Spotify,ഉടായിപ്പ്🤨,negative,0,4,1,undetermined,0,positive,0
9,060896602c6e56de502b7233a6a1ba4408a5837f4f2861...,Google Maps,更新後導航强制變羅馬併音(英文)，對懂日文漢字的華語用户極度不便!,negative,0,3,0,non_english_likely,0,positive,0


## 13. Validation checks

The final checks confirm:

- all 34,601 reviews received a weak label
- the modeling-ready dataset preserves all rows
- direct rating-derived fields are absent
- train and test data have no review-key or normalized-text overlap
- all three classes are present in both partitions
- the confusion matrix reconciles to the test row count
- all exported metrics are finite
- required outputs are created

In [16]:
validation_records = []

def add_validation(check_name, passed, observed):
    validation_records.append(
        {
            "check": check_name,
            "status": "passed" if bool(passed) else "failed",
            "passed": bool(passed),
            "observed": observed,
        }
    )

add_validation(
    "all_reviews_have_weak_label",
    reviews["weak_sentiment_label"].notna().all(),
    int(reviews["weak_sentiment_label"].isna().sum()),
)
add_validation(
    "all_three_labels_present",
    set(reviews["weak_sentiment_label"].unique()) == set(LABEL_ORDER),
    sorted(reviews["weak_sentiment_label"].unique().tolist()),
)
add_validation(
    "modeling_ready_row_count_matches_input",
    len(modeling_ready) == len(reviews),
    len(modeling_ready),
)
add_validation(
    "modeling_ready_review_keys_unique",
    modeling_ready["review_key"].is_unique,
    int(modeling_ready["review_key"].duplicated().sum()),
)
add_validation(
    "score_excluded_from_modeling_ready",
    "score" not in modeling_ready.columns,
    "score" in modeling_ready.columns,
)
add_validation(
    "rating_group_excluded_from_modeling_ready",
    "rating_group" not in modeling_ready.columns,
    "rating_group" in modeling_ready.columns,
)
add_validation(
    "low_rating_flag_excluded_from_modeling_ready",
    "low_rating_flag" not in modeling_ready.columns,
    "low_rating_flag" in modeling_ready.columns,
)
add_validation(
    "train_test_review_key_overlap_zero",
    review_key_overlap == 0,
    review_key_overlap,
)
add_validation(
    "train_test_text_group_overlap_zero",
    text_group_overlap == 0,
    text_group_overlap,
)
add_validation(
    "all_labels_present_in_train",
    set(train_df["weak_sentiment_label"].unique()) == set(LABEL_ORDER),
    sorted(train_df["weak_sentiment_label"].unique().tolist()),
)
add_validation(
    "all_labels_present_in_test",
    set(test_df["weak_sentiment_label"].unique()) == set(LABEL_ORDER),
    sorted(test_df["weak_sentiment_label"].unique().tolist()),
)
add_validation(
    "prediction_count_matches_test_rows",
    len(test_predictions) == len(test_df),
    len(test_predictions),
)
add_validation(
    "confusion_matrix_total_matches_test_rows",
    int(confusion.sum()) == len(test_df),
    int(confusion.sum()),
)
add_validation(
    "model_metrics_are_finite",
    np.isfinite(
        model_metrics[
            ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
        ].to_numpy()
    ).all(),
    "finite" if np.isfinite(
        model_metrics[
            ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
        ].to_numpy()
    ).all() else "non-finite value found",
)
add_validation(
    "rating_mentions_redacted_without_row_loss",
    len(reviews["model_text"]) == len(reviews),
    int(reviews["rating_mention_redacted_flag"].sum()),
)
add_validation(
    "duplicate_identity_flags_remain_zero",
    int(reviews["duplicate_identity_flag"].sum()) == 0,
    int(reviews["duplicate_identity_flag"].sum()),
)

validation_checks = pd.DataFrame(validation_records)
validation_checks.to_csv(
    OUTPUT_DIR / "weak_sentiment_validation_checks_v0.csv",
    index=False,
)

if not validation_checks["passed"].all():
    failed_checks = validation_checks.loc[~validation_checks["passed"]]
    raise ValueError("Validation failed:\n" + failed_checks.to_string(index=False))

display(validation_checks)

,check,status,passed,observed
0,all_reviews_have_weak_label,passed,True,0
1,all_three_labels_present,passed,True,"[negative, neutral, positive]"
2,modeling_ready_row_count_matches_input,passed,True,34601
3,modeling_ready_review_keys_unique,passed,True,0
4,score_excluded_from_modeling_ready,passed,True,False
5,rating_group_excluded_from_modeling_ready,passed,True,False
6,low_rating_flag_excluded_from_modeling_ready,passed,True,False
7,train_test_review_key_overlap_zero,passed,True,0
8,train_test_text_group_overlap_zero,passed,True,0
9,all_labels_present_in_train,passed,True,"[negative, neutral, positive]"


## 14. Document limitations and export metadata

The main limitations are:

1. **Weak-label validity:** ratings are not manually verified sentiment labels.
2. **Neutral-class scarcity:** three-star reviews are only a small share of the dataset.
3. **Rating–text mismatch:** a written review can be mixed, sarcastic, or inconsistent with its numeric rating.
4. **Snapshot scope:** results apply to these 10 apps and the current collected review snapshot.
5. **Language coverage:** the language field is a conservative heuristic, not a trained detector.
6. **Keyword indicators:** issue flags are screening rules rather than verified issue labels.
7. **Repeated text:** exact normalized text is group-separated, but near-duplicate wording can still occur across train and test.
8. **No production claim:** the model has not been externally validated, calibrated, drift-tested, fairness-tested, or deployed.
9. **No app-level generalization test:** app identity is excluded from the model, but all 10 apps still appear in both train and test.
10. **Coefficient interpretation:** linear coefficients show association within this dataset, not causation or universal sentiment meaning.

In [17]:
baseline_metrics = model_metrics.loc[
    model_metrics["model"].eq("class_weighted_linear_svc")
].iloc[0]

metadata = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_stage": "rating_derived_weak_sentiment_baseline_v0",
    "scope": "exploratory_not_production",
    "source_file": str(INPUT_PATH.name),
    "source_sha256": sha256_file(INPUT_PATH),
    "source_rows": int(len(reviews)),
    "apps": int(reviews["app_name"].nunique()),
    "weak_label_mapping": {
        "1-2": "negative",
        "3": "neutral",
        "4-5": "positive",
    },
    "weak_label_warning": (
        "Labels are derived from ratings and are not manually verified sentiment labels."
    ),
    "random_state": RANDOM_STATE,
    "split_method": "first holdout fold from StratifiedGroupKFold(n_splits=5)",
    "group_definition": "SHA-256 of normalized cleaned review text",
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "train_test_text_group_overlap": int(text_group_overlap),
    "rating_mention_redacted_rows": int(
        reviews["rating_mention_redacted_flag"].sum()
    ),
    "model": {
        "name": "LinearSVC",
        "C": 1.0,
        "class_weight": "balanced",
        "hyperparameter_search": False,
        "text_features": "TF-IDF word unigrams and bigrams",
        "max_tfidf_features": 6000,
        "app_identity_used_as_model_input": False,
        "developer_reply_used_as_model_input": False,
        "rating_fields_used_as_model_input": False,
    },
    "evaluation": {
        "accuracy": float(baseline_metrics["accuracy"]),
        "balanced_accuracy": float(baseline_metrics["balanced_accuracy"]),
        "macro_f1": float(baseline_metrics["macro_f1"]),
        "weighted_f1": float(baseline_metrics["weighted_f1"]),
    },
    "validation_checks_passed": int(validation_checks["passed"].sum()),
    "validation_checks_failed": int((~validation_checks["passed"]).sum()),
}

metadata_path = OUTPUT_DIR / "weak_sentiment_baseline_metadata_v0.json"
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(metadata, indent=2))

{
  "generated_at_utc": "2026-07-25T19:49:35.603818+00:00",
  "project_stage": "rating_derived_weak_sentiment_baseline_v0",
  "scope": "exploratory_not_production",
  "source_file": "review_features_v0.csv",
  "source_sha256": "7e65a3d282b7121012a940c931bed0a10f32ebf83d8744d1021545dbb87fe2fc",
  "source_rows": 34601,
  "apps": 10,
  "weak_label_mapping": {
    "1-2": "negative",
    "3": "neutral",
    "4-5": "positive"
  },
  "weak_label_warning": "Labels are derived from ratings and are not manually verified sentiment labels.",
  "random_state": 42,
  "split_method": "first holdout fold from StratifiedGroupKFold(n_splits=5)",
  "group_definition": "SHA-256 of normalized cleaned review text",
  "train_rows": 27681,
  "test_rows": 6920,
  "train_test_text_group_overlap": 0,
  "rating_mention_redacted_rows": 415,
  "model": {
    "name": "LinearSVC",
    "C": 1.0,
    "class_weight": "balanced",
    "hyperparameter_search": false,
    "text_features": "TF-IDF word unigrams and bigrams",

## 15. Generate the report, model card, and README update

The generated documentation records:

- weak-label definition
- class distribution
- feature-pattern findings
- leakage controls
- split and model design
- baseline evaluation
- limitations
- intended and non-intended uses

In [18]:
distribution_lookup = class_distribution.set_index(
    "weak_sentiment_label"
).to_dict("index")
pattern_lookup = feature_patterns.set_index(
    "weak_sentiment_label"
).to_dict("index")
class_report_lookup = classification_report_df.set_index("class").to_dict("index")

report_text = f"""# Google Play Rating-Derived Weak Sentiment Baseline v0 Report

## Objective

This downstream baseline uses the validated Feature Engineering v0 review table to create rating-derived weak sentiment labels, compare selected feature patterns, prepare a clean modeling-ready dataset, and test one transparent linear model.

The work is exploratory and is not a production sentiment system.

## Weak-label definition

- 1–2 stars: `negative`
- 3 stars: `neutral`
- 4–5 stars: `positive`

These labels are derived from source ratings. They are not manually reviewed ground-truth sentiment annotations.

## Data

- Review rows: {len(reviews):,}
- Apps: {reviews['app_name'].nunique()}
- Training rows: {len(train_df):,}
- Test rows: {len(test_df):,}
- Exact normalized-text groups shared across train and test: {text_group_overlap}
- Reviews with explicit written star-rating expressions redacted: {int(reviews['rating_mention_redacted_flag'].sum()):,}

## Class distribution

| Weak label | Reviews | Share |
|---|---:|---:|
| Negative | {distribution_lookup['negative']['review_count']:,} | {distribution_lookup['negative']['review_share_pct']:.2f}% |
| Neutral | {distribution_lookup['neutral']['review_count']:,} | {distribution_lookup['neutral']['review_share_pct']:.2f}% |
| Positive | {distribution_lookup['positive']['review_count']:,} | {distribution_lookup['positive']['review_share_pct']:.2f}% |

The neutral class is substantially smaller than the negative and positive classes, so accuracy alone is not a sufficient evaluation measure.

## Selected feature patterns

| Pattern | Negative | Neutral | Positive |
|---|---:|---:|---:|
| Median review characters | {pattern_lookup['negative']['median_review_char_count']:.0f} | {pattern_lookup['neutral']['median_review_char_count']:.0f} | {pattern_lookup['positive']['median_review_char_count']:.0f} |
| Median review words | {pattern_lookup['negative']['median_review_word_count']:.0f} | {pattern_lookup['neutral']['median_review_word_count']:.0f} | {pattern_lookup['positive']['median_review_word_count']:.0f} |
| Short-review share | {pattern_lookup['negative']['short_review_share']:.2%} | {pattern_lookup['neutral']['short_review_share']:.2%} | {pattern_lookup['positive']['short_review_share']:.2%} |
| Low-signal share | {pattern_lookup['negative']['low_signal_share']:.2%} | {pattern_lookup['neutral']['low_signal_share']:.2%} | {pattern_lookup['positive']['low_signal_share']:.2%} |
| Repeated-text share | {pattern_lookup['negative']['repeated_text_share']:.2%} | {pattern_lookup['neutral']['repeated_text_share']:.2%} | {pattern_lookup['positive']['repeated_text_share']:.2%} |
| Developer-reply share | {pattern_lookup['negative']['developer_reply_share']:.2%} | {pattern_lookup['neutral']['developer_reply_share']:.2%} | {pattern_lookup['positive']['developer_reply_share']:.2%} |
| Any issue-indicator share | {pattern_lookup['negative']['any_issue_indicator_share']:.2%} | {pattern_lookup['neutral']['any_issue_indicator_share']:.2%} | {pattern_lookup['positive']['any_issue_indicator_share']:.2%} |

Negative reviews are longer and more likely to match at least one issue-keyword category. Positive reviews are much more likely to be short, low-signal, or repeated. These are descriptive associations in the collected snapshot and should not be treated as causal findings.

## Leakage controls

The target is derived from rating, so `score`, `rating_group`, and `low_rating_flag` are excluded from the modeling-ready dataset and model inputs.

Additional controls:

- app identity is excluded from the model to reduce app-specific rating-prior shortcuts
- developer-reply availability is excluded because it may be a post-review signal
- collection dates, run IDs, app versions, and source-lag fields are excluded
- exact normalized text groups are kept in only one split
- explicit star-rating expressions are replaced with a generic token before TF-IDF generation

## Baseline model

- Model: class-weighted `LinearSVC`
- Regularization: fixed `C=1.0`
- Hyperparameter search: none
- Text representation: TF-IDF word unigrams and bigrams
- Additional inputs: text-length, low-signal, repeated-text, language-group, and keyword issue-indicator features
- Reference: majority-class dummy classifier

## Evaluation

| Model | Accuracy | Balanced accuracy | Macro F1 | Weighted F1 |
|---|---:|---:|---:|---:|
| Majority-class reference | {model_metrics.iloc[0]['accuracy']:.4f} | {model_metrics.iloc[0]['balanced_accuracy']:.4f} | {model_metrics.iloc[0]['macro_f1']:.4f} | {model_metrics.iloc[0]['weighted_f1']:.4f} |
| Class-weighted LinearSVC | {model_metrics.iloc[1]['accuracy']:.4f} | {model_metrics.iloc[1]['balanced_accuracy']:.4f} | {model_metrics.iloc[1]['macro_f1']:.4f} | {model_metrics.iloc[1]['weighted_f1']:.4f} |

| Class | Precision | Recall | F1 | Test support |
|---|---:|---:|---:|---:|
| Negative | {class_report_lookup['negative']['precision']:.4f} | {class_report_lookup['negative']['recall']:.4f} | {class_report_lookup['negative']['f1_score']:.4f} | {class_report_lookup['negative']['support']:,} |
| Neutral | {class_report_lookup['neutral']['precision']:.4f} | {class_report_lookup['neutral']['recall']:.4f} | {class_report_lookup['neutral']['f1_score']:.4f} | {class_report_lookup['neutral']['support']:,} |
| Positive | {class_report_lookup['positive']['precision']:.4f} | {class_report_lookup['positive']['recall']:.4f} | {class_report_lookup['positive']['f1_score']:.4f} | {class_report_lookup['positive']['support']:,} |

The linear model improves substantially over the majority-class reference on balanced accuracy and macro F1. Performance is strongest for positive and negative reviews. Neutral-review performance remains weak, which is consistent with the small neutral class and the ambiguity of a rating-derived three-star label.

## Interpretation

The exported coefficient table shows the strongest directional associations for each class. These weights are useful for model inspection, but they are not causal explanations or universal sentiment rules.

## Limitations

1. Ratings are weak labels rather than manually verified sentiment.
2. Three-star reviews are rare and may contain mixed or ambiguous sentiment.
3. Review text and rating can disagree.
4. The snapshot covers 10 selected Google Play apps and one accumulated collection period.
5. Language and issue indicators are heuristics.
6. Exact duplicate text is group-separated, but near-duplicate phrasing may cross partitions.
7. All 10 apps appear in both training and test data; cross-app generalization is not established.
8. The model is not probability-calibrated and has not been externally validated.
9. No production monitoring, drift testing, fairness testing, or deployment work is included.
10. Coefficients reflect associations within this dataset, not causation.

## Conclusion

The baseline satisfies the current exploratory objective: it creates a clean leakage-controlled dataset, documents the weak-label assumption, shows class and feature patterns, and provides one transparent model with reproducible evaluation outputs. It should be treated as a starting point for error analysis and label-quality review rather than as a production sentiment classifier.
"""

report_path = REPORT_DIR / "google_play_weak_sentiment_baseline_v0_report.md"
report_path.write_text(report_text, encoding="utf-8")

model_card_text = f"""# Weak Sentiment Baseline v0 Model Card

## Model

Class-weighted linear support vector classifier (`LinearSVC`, `C=1.0`) using TF-IDF word unigrams and bigrams plus a small set of deterministic review-level features.

## Intended use

- exploratory comparison of rating-derived weak sentiment groups
- baseline error analysis
- demonstration of a downstream modeling-ready workflow
- identification of label-quality and class-imbalance issues

## Not intended for

- production sentiment classification
- customer-level decisions
- automated moderation
- app-quality ranking
- ground-truth sentiment labeling
- performance claims beyond the current 10-app snapshot

## Target

- 1–2 stars = negative
- 3 stars = neutral
- 4–5 stars = positive

The target is a weak label derived from rating, not a manually verified sentiment label.

## Leakage controls

- `score`, `rating_group`, and `low_rating_flag` excluded
- explicit star-rating expressions redacted to a generic token
- exact normalized text grouped into one train/test partition
- app identity and developer-reply availability excluded from model inputs
- run and collection fields excluded

## Evaluation snapshot

- Test rows: {len(test_df):,}
- Accuracy: {baseline_metrics['accuracy']:.4f}
- Balanced accuracy: {baseline_metrics['balanced_accuracy']:.4f}
- Macro F1: {baseline_metrics['macro_f1']:.4f}
- Weighted F1: {baseline_metrics['weighted_f1']:.4f}

## Main risk

The neutral class is small and weakly defined. Neutral recall and F1 are substantially lower than positive and negative performance.

## Required interpretation

Outputs represent associations with rating-derived labels in the current dataset. They do not establish true sentiment, causation, or production readiness.
"""

model_card_path = REPORT_DIR / "weak_sentiment_baseline_v0_model_card.md"
model_card_path.write_text(model_card_text, encoding="utf-8")

readme_update_text = f"""## Rating-derived weak sentiment baseline v0

### Objective

This downstream exploratory layer uses the validated `review_features_v0.csv` table to derive weak sentiment labels from source ratings:

- 1–2 stars: `negative`
- 3 stars: `neutral`
- 4–5 stars: `positive`

The labels are weak labels and are not manually verified sentiment annotations.

### Leakage controls

The model excludes `score`, `rating_group`, and `low_rating_flag`. It also excludes app identity, developer-reply availability, collection dates, run IDs, app versions, and source-lag fields from model inputs. Exact normalized text groups are assigned to only one data split, and explicit written star-rating expressions are replaced with a generic token before modeling.

### Data and class distribution

- 34,601 review rows
- 10 apps
- {len(train_df):,} training rows
- {len(test_df):,} test rows
- negative: {distribution_lookup['negative']['review_count']:,} ({distribution_lookup['negative']['review_share_pct']:.2f}%)
- neutral: {distribution_lookup['neutral']['review_count']:,} ({distribution_lookup['neutral']['review_share_pct']:.2f}%)
- positive: {distribution_lookup['positive']['review_count']:,} ({distribution_lookup['positive']['review_share_pct']:.2f}%)
- exact normalized-text overlap between train and test: 0

### Exploratory feature patterns

Negative reviews are longer and more likely to match issue-keyword indicators. Positive reviews are more likely to be short, low-signal, or repeated. These are descriptive patterns in the current snapshot rather than causal conclusions.

### Baseline model

A class-weighted linear support vector classifier uses TF-IDF word unigrams and bigrams plus selected deterministic review features. The model uses fixed `C=1.0` and no hyperparameter search.

| Model | Accuracy | Balanced accuracy | Macro F1 | Weighted F1 |
|---|---:|---:|---:|---:|
| Majority-class reference | {model_metrics.iloc[0]['accuracy']:.4f} | {model_metrics.iloc[0]['balanced_accuracy']:.4f} | {model_metrics.iloc[0]['macro_f1']:.4f} | {model_metrics.iloc[0]['weighted_f1']:.4f} |
| Class-weighted LinearSVC | {model_metrics.iloc[1]['accuracy']:.4f} | {model_metrics.iloc[1]['balanced_accuracy']:.4f} | {model_metrics.iloc[1]['macro_f1']:.4f} | {model_metrics.iloc[1]['weighted_f1']:.4f} |

The model performs substantially better than the majority-class reference on balanced accuracy and macro F1. Neutral-class performance remains limited because three-star reviews are rare and may express mixed sentiment.

### Main outputs

```text
notebooks/
└── Google_Play_Weak_Sentiment_Baseline_v0.ipynb

outputs/
├── modeling_ready_weak_sentiment_v0.csv
├── weak_sentiment_source_validation_v0.csv
├── weak_sentiment_class_distribution_v0.csv
├── weak_sentiment_feature_patterns_v0.csv
├── weak_sentiment_issue_patterns_v0.csv
├── weak_sentiment_app_distribution_v0.csv
├── weak_sentiment_model_metrics_v0.csv
├── weak_sentiment_classification_report_v0.csv
├── weak_sentiment_confusion_matrix_v0.csv
├── weak_sentiment_top_coefficients_v0.csv
├── weak_sentiment_test_predictions_sample_v0.csv
├── weak_sentiment_validation_checks_v0.csv
├── weak_sentiment_baseline_metadata_v0.json
└── weak_sentiment_output_manifest_v0.csv

reports/
├── figures/
│   ├── weak_sentiment_class_distribution_v0.png
│   ├── weak_sentiment_feature_patterns_v0.png
│   └── weak_sentiment_confusion_matrix_v0.png
├── google_play_weak_sentiment_baseline_v0_report.md
├── weak_sentiment_baseline_v0_model_card.md
└── README_weak_sentiment_baseline_v0_update.md
```

### Limitations

This is an exploratory baseline, not a production system. Ratings are weak labels, the neutral class is small, text and rating can disagree, heuristic fields can be noisy, and cross-app or future-time generalization has not been established.
"""

readme_update_path = REPORT_DIR / "README_weak_sentiment_baseline_v0_update.md"
readme_update_path.write_text(readme_update_text, encoding="utf-8")

print("Saved report:", report_path)
print("Saved model card:", model_card_path)
print("Saved README update:", readme_update_path)

Saved report: /mnt/data/google_play_weak_sentiment_baseline_v0_complete_github_package/reports/google_play_weak_sentiment_baseline_v0_report.md
Saved model card: /mnt/data/google_play_weak_sentiment_baseline_v0_complete_github_package/reports/weak_sentiment_baseline_v0_model_card.md
Saved README update: /mnt/data/google_play_weak_sentiment_baseline_v0_complete_github_package/reports/README_weak_sentiment_baseline_v0_update.md


## 16. Create the output manifest and final summary

The manifest records file size and SHA-256 for all baseline outputs generated by this notebook.

In [19]:
manifest_targets = [
    OUTPUT_DIR / "weak_sentiment_source_validation_v0.csv",
    OUTPUT_DIR / "weak_sentiment_class_distribution_v0.csv",
    OUTPUT_DIR / "weak_sentiment_feature_patterns_v0.csv",
    OUTPUT_DIR / "weak_sentiment_issue_patterns_v0.csv",
    OUTPUT_DIR / "weak_sentiment_app_distribution_v0.csv",
    OUTPUT_DIR / "modeling_ready_weak_sentiment_v0.csv",
    OUTPUT_DIR / "weak_sentiment_model_metrics_v0.csv",
    OUTPUT_DIR / "weak_sentiment_classification_report_v0.csv",
    OUTPUT_DIR / "weak_sentiment_confusion_matrix_v0.csv",
    OUTPUT_DIR / "weak_sentiment_top_coefficients_v0.csv",
    OUTPUT_DIR / "weak_sentiment_test_predictions_sample_v0.csv",
    OUTPUT_DIR / "weak_sentiment_validation_checks_v0.csv",
    OUTPUT_DIR / "weak_sentiment_baseline_metadata_v0.json",
    FIGURE_DIR / "weak_sentiment_class_distribution_v0.png",
    FIGURE_DIR / "weak_sentiment_feature_patterns_v0.png",
    FIGURE_DIR / "weak_sentiment_confusion_matrix_v0.png",
    REPORT_DIR / "google_play_weak_sentiment_baseline_v0_report.md",
    REPORT_DIR / "weak_sentiment_baseline_v0_model_card.md",
    REPORT_DIR / "README_weak_sentiment_baseline_v0_update.md",
]

manifest_rows = []
for file_path in manifest_targets:
    if not file_path.exists() or file_path.stat().st_size == 0:
        raise FileNotFoundError(f"Required output missing or empty: {file_path}")

    manifest_rows.append(
        {
            "relative_path": str(file_path.relative_to(REPO_ROOT)),
            "size_bytes": file_path.stat().st_size,
            "sha256": sha256_file(file_path),
        }
    )

output_manifest = pd.DataFrame(manifest_rows)
output_manifest_path = OUTPUT_DIR / "weak_sentiment_output_manifest_v0.csv"
output_manifest.to_csv(output_manifest_path, index=False)

print("All source, leakage, split, feature, model, and output checks passed.")
print(f"Review rows: {len(reviews):,}")
print(f"Train/test rows: {len(train_df):,} / {len(test_df):,}")
print(f"Validation checks passed: {validation_checks['passed'].sum()}")
print(f"Baseline accuracy: {baseline_metrics['accuracy']:.4f}")
print(f"Baseline balanced accuracy: {baseline_metrics['balanced_accuracy']:.4f}")
print(f"Baseline macro F1: {baseline_metrics['macro_f1']:.4f}")
print(f"Baseline weighted F1: {baseline_metrics['weighted_f1']:.4f}")
print("Output manifest:", output_manifest_path)

display(output_manifest)

All source, leakage, split, feature, model, and output checks passed.
Review rows: 34,601
Train/test rows: 27,681 / 6,920
Validation checks passed: 16
Baseline accuracy: 0.8208
Baseline balanced accuracy: 0.5867
Baseline macro F1: 0.5899
Baseline weighted F1: 0.8196
Output manifest: /mnt/data/google_play_weak_sentiment_baseline_v0_complete_github_package/outputs/weak_sentiment_output_manifest_v0.csv


,relative_path,size_bytes,sha256
0,outputs/weak_sentiment_source_validation_v0.csv,517,8669b2a6fa6d297ddaecf185ac99fa7fbdfa670cd41063...
1,outputs/weak_sentiment_class_distribution_v0.csv,220,95e4dff180accf4fa891d3de739d965127e6ceadf71035...
2,outputs/weak_sentiment_feature_patterns_v0.csv,1136,701cd4462562e82c3cd7dcd6bc497de14cda94f961bfcd...
3,outputs/weak_sentiment_issue_patterns_v0.csv,644,3f068663c6368c2cab7013862b088d8497955c1c4769af...
4,outputs/weak_sentiment_app_distribution_v0.csv,957,5d42b5121fa18f78b7e438bde03c8d38c93ad7c3bcbd64...
5,outputs/modeling_ready_weak_sentiment_v0.csv,10347072,53fc5b63a797ba22a063ca6c67a5062d8f3d84441112ee...
6,outputs/weak_sentiment_model_metrics_v0.csv,277,a117c248e0eee51e9d7a6da349bbd6a110f0c61611b3d0...
7,outputs/weak_sentiment_classification_report_v...,409,b52ee9ceaf77e41fdace0fd2c15863cb9044d7939b40f3...
8,outputs/weak_sentiment_confusion_matrix_v0.csv,141,654210ff830cfd3b29c9a8f784711ce73a0b064da7ae09...
9,outputs/weak_sentiment_top_coefficients_v0.csv,7131,58fe66f7ed8cb8440f3c278d09ee4b5f0d22379021bc4f...
